# 1. Hilbert Transform-Based Active-Window Extraction

This notebook extracts the active response window in which the vehicle-induced bridge response is clearly observed. Welch PSD and frequency-band calculations are not performed in this step. Instead, STR sensor responses are processed using the Hilbert transform to identify sensor-level start and end times for the active response interval.


## 1.1 STR Sensor-Level Active-Window Extraction

Because the vehicle reaches Span 1 and Span 2 at different times, the response timing is expected to vary across sensors within the same event. The 12 STR sensors are arranged in a `3 columns x 4 rows` layout, with the upper two rows corresponding to S1 and the lower two rows corresponding to S2.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'aquinas_common.py').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Project root containing utils/aquinas_common.py was not found.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import hilbert
from IPython.display import Markdown, display

from utils.aquinas_spectral_tables import ensure_spectral_tables
from utils.aquinas_common import (
    AXIS_LABEL_FONTSIZE,
    TICK_LABEL_FONTSIZE,
    LEGEND_FONTSIZE,
    PLOT_TITLE_FONTSIZE,
    EPS,
    bandpass_strain,
    choose_reference_event,
    compact_display_path,
    configure_notebook_style,
    fill_signal,
    method_sensor_index,
    parse_alias,
    read_event_hdf5,
    sensor_hilbert_window,
    timing_bounds_for_event,
    valid_time_bounds,
)

configure_notebook_style(dpi=150, grid=True)
ensure_spectral_tables(require_psd=False)

STR_COLOR = '#2563EB'
ACC_COLOR = '#16A34A'
HILBERT_COLOR = '#6B7280'
PANEL_TITLE_FONTSIZE = 13

CLEAN_SINGLE_DIP_EVENT_PATH = (
    PROJECT_ROOT / 'EWSHM_dataset_preprocessed_event_level'
    / 'AQUINAS_SET2_2023_04'
    / 'OLD'
    / 'PART020'
    / 'AQUINAS_SET2_OLD_2023_04_PART020_EVENT028_ID001266.hdf5'
)

if CLEAN_SINGLE_DIP_EVENT_PATH.exists():
    score_table_path = PROJECT_ROOT / 'AutoResearch_generated_method' / 'tables_v2' / 'event_spectral_health_scores_v2.csv'
    score_table = pd.read_csv(score_table_path, low_memory=False)
    matched = score_table[score_table['path'].eq(str(CLEAN_SINGLE_DIP_EVENT_PATH))]
    if matched.empty:
        reference_event = pd.Series({
            'path': str(CLEAN_SINGLE_DIP_EVENT_PATH),
            'event_health_score_v2': np.nan,
        })
    else:
        reference_event = matched.iloc[0]
else:
    reference_event = choose_reference_event()

event_path = reference_event['path']
values, mask, aliases, time_grid = read_event_hdf5(event_path)

event_bounds = timing_bounds_for_event(event_path)
if event_bounds is None:
    event_start, event_end = valid_time_bounds(mask, time_grid)
    window_source = 'valid signal range'
else:
    event_start, event_end = event_bounds
    window_source = 'event-level Hilbert timing diagnostics'

meta = pd.DataFrame([parse_alias(alias) for alias in aliases])
meta['hdf5_row_index'] = np.arange(len(meta), dtype=int)
meta['hdf5_sensor_index'] = meta['hdf5_row_index'] + 1
meta['sensor_index'] = meta['sensor_alias'].map(method_sensor_index).astype(int)
str_meta = meta[meta['quantity'].eq('STR')].copy()
if str_meta.empty:
    raise RuntimeError('No strain sensor was found in the selected event.')

str_meta['_span'] = str_meta['span'].map({'S1': 0, 'S2': 1}).fillna(9)
str_meta['_side'] = str_meta['side'].map({'UP': 0, 'DO': 1}).fillna(9)
str_meta['_location'] = str_meta['location'].map({'INF': 0, 'SHE': 1, 'SUP': 2}).fillna(9)
str_meta = str_meta.sort_values(['sensor_index']).reset_index(drop=True)

display(Markdown(f'''
**Selected event**

- event path: `{compact_display_path(event_path)}`
- reference event health score: `{reference_event['event_health_score_v2']:.2f} / 100`
- event-level timing source: `{window_source}`
- event-level active window: `{event_start:.2f}` s to `{event_end:.2f}` s
- STR channels shown: `{len(str_meta)}`
'''))

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(14.8, 10.4), sharex=True, sharey=True)
axes = axes.ravel()
sensor_windows = []

for panel_idx, ax in enumerate(axes):
    if panel_idx >= len(str_meta):
        ax.axis('off')
        continue

    row = str_meta.iloc[panel_idx]
    idx = int(row['hdf5_row_index'])
    alias = row['sensor_alias']
    valid_i = mask[idx].astype(bool)

    signal = values[idx].astype(float)
    centered = signal - np.nanmedian(signal[valid_i])
    signal_scale = np.nanpercentile(np.abs(centered[valid_i]), 95) if valid_i.any() else np.nan
    if not np.isfinite(signal_scale) or signal_scale <= EPS:
        signal_scale = 1.0
    display_signal = centered / signal_scale
    display_signal[~valid_i] = np.nan

    y_fill = fill_signal(values[idx], valid_i)
    y_band = bandpass_strain(y_fill)
    envelope = np.abs(hilbert(y_band))
    sensor_start, sensor_end, envelope_smooth = sensor_hilbert_window(envelope, valid_i, time_grid)
    sensor_windows.append({'sensor': alias, 'span': row['span'], 'start_s': sensor_start, 'end_s': sensor_end})

    envelope_scale = np.nanpercentile(envelope_smooth[valid_i], 95) if valid_i.any() else np.nan
    if not np.isfinite(envelope_scale) or envelope_scale <= EPS:
        envelope_scale = 1.0
    envelope_plot = envelope_smooth / (envelope_scale + EPS)

    ax.plot(time_grid, display_signal, color=STR_COLOR, lw=0.95, label='STR')
    ax.plot(time_grid, envelope_plot, color=HILBERT_COLOR, lw=1.10, alpha=0.86, label='Hilbert response')

    if np.isfinite(sensor_start) and np.isfinite(sensor_end):
        ax.axvspan(sensor_start, sensor_end, color='#F59E0B', alpha=0.17, linewidth=0)
        ax.axvline(sensor_start, color='#D97706', lw=1.05)
        ax.axvline(sensor_end, color='#D97706', lw=1.05)
        ax.text(
            0.03,
            0.93,
            f'start {sensor_start:.2f} s\nend {sensor_end:.2f} s',
            transform=ax.transAxes,
            ha='left',
            va='top',
            fontsize=7.8,
            color='#7C2D12',
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.72, pad=2.0),
        )

    ax.set_title(alias, fontsize=TICK_LABEL_FONTSIZE)
    ax.grid(True, alpha=0.18)
    ax.tick_params(axis='both', labelsize=TICK_LABEL_FONTSIZE)

finite_starts = [w['start_s'] for w in sensor_windows if np.isfinite(w['start_s'])]
finite_ends = [w['end_s'] for w in sensor_windows if np.isfinite(w['end_s'])]
if finite_starts and finite_ends:
    x_min = max(float(time_grid[0]), min(finite_starts) - 1.2)
    x_max = min(float(time_grid[-1]), max(finite_ends) + 1.8)
else:
    x_min = max(float(time_grid[0]), event_start - 2.0)
    x_max = min(float(time_grid[-1]), event_end + 3.0)

for ax in axes:
    if ax.has_data():
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(-3.5, 3.5)

for ax in axes[9:12]:
    ax.set_xlabel('Time (s)', fontsize=AXIS_LABEL_FONTSIZE)
for ax in axes[::3]:
    ax.set_ylabel('Scaled response', fontsize=AXIS_LABEL_FONTSIZE)

fig.text(0.012, 0.705, 'S1', ha='center', va='center', rotation=90, fontsize=15, fontweight='bold', color='0.25')
fig.text(0.012, 0.325, 'S2', ha='center', va='center', rotation=90, fontsize=15, fontweight='bold', color='0.25')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, frameon=True, fontsize=LEGEND_FONTSIZE)
fig.suptitle('Sensor-specific active windows from STR Hilbert responses', fontsize=PLOT_TITLE_FONTSIZE, y=0.98)
fig.tight_layout(rect=(0.02, 0.065, 1, 0.955))

plt.show()

sensor_window_table = pd.DataFrame(sensor_windows)
display(sensor_window_table)

## 1.2 Final Event-Level Active-Window Verification

The previous figure shows active windows extracted separately for each STR sensor. This section combines the STR responses and displays the final event-level active window used in the subsequent analysis, using the same `3 columns x 4 rows` layout.


In [ ]:
if finite_starts and finite_ends:
    final_start = float(min(finite_starts))
    final_end = float(max(finite_ends))
else:
    final_start = float(event_start)
    final_end = float(event_end)

fig, axes = plt.subplots(4, 3, figsize=(14.8, 10.4), sharex=True, sharey=True)
axes = axes.ravel()

for panel_idx, ax in enumerate(axes):
    if panel_idx >= len(str_meta):
        ax.axis('off')
        continue

    row = str_meta.iloc[panel_idx]
    idx = int(row['hdf5_row_index'])
    alias = row['sensor_alias']
    valid_i = mask[idx].astype(bool)

    signal = values[idx].astype(float)
    centered = signal - np.nanmedian(signal[valid_i])
    signal_scale = np.nanpercentile(np.abs(centered[valid_i]), 95) if valid_i.any() else np.nan
    if not np.isfinite(signal_scale) or signal_scale <= EPS:
        signal_scale = 1.0
    display_signal = centered / signal_scale
    display_signal[~valid_i] = np.nan

    y_fill = fill_signal(values[idx], valid_i)
    y_band = bandpass_strain(y_fill)
    envelope = np.abs(hilbert(y_band))
    _, _, envelope_smooth = sensor_hilbert_window(envelope, valid_i, time_grid)

    envelope_scale = np.nanpercentile(envelope_smooth[valid_i], 95) if valid_i.any() else np.nan
    if not np.isfinite(envelope_scale) or envelope_scale <= EPS:
        envelope_scale = 1.0
    envelope_plot = envelope_smooth / (envelope_scale + EPS)

    ax.plot(time_grid, display_signal, color=STR_COLOR, lw=0.95, label='STR')
    ax.plot(time_grid, envelope_plot, color=HILBERT_COLOR, lw=1.10, alpha=0.86, label='Hilbert response')
    ax.axvspan(final_start, final_end, color='#4C78A8', alpha=0.17, linewidth=0)
    ax.axvline(final_start, color='#2563EB', lw=1.20)
    ax.axvline(final_end, color='#2563EB', lw=1.20)
    ax.text(
        0.03,
        0.93,
        f'final start {final_start:.2f} s\nfinal end {final_end:.2f} s',
        transform=ax.transAxes,
        ha='left',
        va='top',
        fontsize=7.8,
        color='#1E3A8A',
        fontweight='bold',
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.72, pad=2.0),
    )
    ax.set_title(alias, fontsize=TICK_LABEL_FONTSIZE)
    ax.grid(True, alpha=0.18)
    ax.tick_params(axis='both', labelsize=TICK_LABEL_FONTSIZE)

x_min = max(float(time_grid[0]), final_start - 1.4)
x_max = min(float(time_grid[-1]), final_end + 1.8)
for ax in axes:
    if ax.has_data():
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(-3.5, 3.5)

for ax in axes[9:12]:
    ax.set_xlabel('Time (s)', fontsize=AXIS_LABEL_FONTSIZE)
for ax in axes[::3]:
    ax.set_ylabel('Scaled response', fontsize=AXIS_LABEL_FONTSIZE)

fig.text(0.012, 0.705, 'S1', ha='center', va='center', rotation=90, fontsize=15, fontweight='bold', color='0.25')
fig.text(0.012, 0.325, 'S2', ha='center', va='center', rotation=90, fontsize=15, fontweight='bold', color='0.25')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, frameon=True, fontsize=LEGEND_FONTSIZE)
fig.suptitle('Final event-level active window applied to all STR responses', fontsize=PLOT_TITLE_FONTSIZE, y=0.98)
fig.tight_layout(rect=(0.02, 0.065, 1, 0.955))

plt.show()

## 1.3 ACC Response Verification in the Same Active Window

For reference, the ACC sensor responses are displayed using the final STR-based active window. This verifies how the acceleration responses appear within the same time interval after the active window has been defined from the strain responses.


In [ ]:
acc_meta = meta[meta['quantity'].eq('ACC')].copy()
if acc_meta.empty:
    raise RuntimeError('No acceleration sensor was found in the selected event.')

acc_meta['_span'] = acc_meta['span'].map({'S1': 0, 'S2': 1}).fillna(9)
acc_meta['_location'] = acc_meta['location'].map({'INT': 0, 'MID': 1}).fillna(9)
acc_meta['_side'] = acc_meta['side'].map({'UP': 0, 'DO': 1}).fillna(9)
acc_meta['_axis'] = acc_meta['axis'].map({'Y': 0, 'Z': 1}).fillna(9)
acc_meta = acc_meta.sort_values(['sensor_index']).reset_index(drop=True)

fig, axes = plt.subplots(4, 3, figsize=(14.8, 10.4), sharex=True, sharey=True)
axes = axes.ravel()
final_active = (time_grid >= final_start) & (time_grid <= final_end)

for panel_idx, ax in enumerate(axes):
    if panel_idx >= len(acc_meta):
        ax.axis('off')
        continue

    row = acc_meta.iloc[panel_idx]
    idx = int(row['hdf5_row_index'])
    alias = row['sensor_alias']
    valid_i = mask[idx].astype(bool)

    signal = values[idx].astype(float)
    centered = signal - np.nanmedian(signal[valid_i])
    scale_region = valid_i & final_active
    signal_scale = np.nanpercentile(np.abs(centered[scale_region]), 95) if scale_region.any() else np.nan
    if not np.isfinite(signal_scale) or signal_scale <= EPS:
        signal_scale = np.nanpercentile(np.abs(centered[valid_i]), 95) if valid_i.any() else np.nan
    if not np.isfinite(signal_scale) or signal_scale <= EPS:
        signal_scale = 1.0
    display_signal = centered / signal_scale
    display_signal[~valid_i] = np.nan

    ax.plot(time_grid, display_signal, color=ACC_COLOR, lw=0.95, label='ACC')
    ax.axvspan(final_start, final_end, color='#4C78A8', alpha=0.17, linewidth=0)
    ax.axvline(final_start, color='#2563EB', lw=1.20)
    ax.axvline(final_end, color='#2563EB', lw=1.20)
    ax.text(
        0.03,
        0.93,
        f'final start {final_start:.2f} s\nfinal end {final_end:.2f} s',
        transform=ax.transAxes,
        ha='left',
        va='top',
        fontsize=7.8,
        color='#1E3A8A',
        fontweight='bold',
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.72, pad=2.0),
    )
    ax.set_title(alias, fontsize=TICK_LABEL_FONTSIZE)
    ax.grid(True, alpha=0.18)
    ax.tick_params(axis='both', labelsize=TICK_LABEL_FONTSIZE)

x_min = max(float(time_grid[0]), final_start - 1.4)
x_max = min(float(time_grid[-1]), final_end + 1.8)
for ax in axes:
    if ax.has_data():
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(-3.5, 3.5)

for ax in axes[9:12]:
    ax.set_xlabel('Time (s)', fontsize=AXIS_LABEL_FONTSIZE)
for ax in axes[::3]:
    ax.set_ylabel('Scaled response', fontsize=AXIS_LABEL_FONTSIZE)

fig.text(0.012, 0.705, 'S1', ha='center', va='center', rotation=90, fontsize=15, fontweight='bold', color='0.25')
fig.text(0.012, 0.325, 'S2', ha='center', va='center', rotation=90, fontsize=15, fontweight='bold', color='0.25')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=1, frameon=True, fontsize=LEGEND_FONTSIZE)
fig.suptitle('ACC responses under the final STR-based active window', fontsize=PLOT_TITLE_FONTSIZE, y=0.98)
fig.tight_layout(rect=(0.02, 0.065, 1, 0.955))

plt.show()

## 1.4 Summary

The purpose of this step is to isolate the time interval in which the vehicle-induced response is actually observed, rather than using the entire triggered record. The final event-level active window is fixed for the following PSD analysis. The next notebook uses only this active-window interval to compute Welch PSD features and construct spectral response vectors.


## 1.5 Representative STR and ACC response under the same active window

The final active window is overlaid on one representative STR response and one representative ACC response.

In [ ]:
def _actual_signal_for_display(sensor_row):
    idx = int(sensor_row['hdf5_row_index'])
    valid_i = mask[idx].astype(bool)
    signal = values[idx].astype(float)
    baseline = np.nanmedian(signal[valid_i]) if valid_i.any() else np.nanmedian(signal)
    display_signal = signal - baseline
    display_signal[~valid_i] = np.nan
    return display_signal


def _choose_representative(meta_table, preferred_aliases):
    for alias in preferred_aliases:
        rows = meta_table[meta_table['sensor_alias'].eq(alias)]
        if not rows.empty:
            return rows.iloc[0]
    return meta_table.iloc[0]


def _actual_ylim(signal_a, signal_b):
    finite_values = []
    for signal in [signal_a, signal_b]:
        y = np.asarray(signal, dtype=float)
        finite = np.isfinite(y)
        if finite.any():
            finite_values.append(y[finite])
    if not finite_values:
        return -1.0, 1.0
    flat = np.concatenate(finite_values)
    low, high = np.nanpercentile(flat, [0.5, 99.5])
    if not np.isfinite(low) or not np.isfinite(high) or low == high:
        amp = max(1.0, float(np.nanmax(np.abs(flat))))
        return -amp, amp
    pad = 0.12 * (high - low)
    return float(low - pad), float(high + pad)

representative_str = _choose_representative(
    str_meta,
    ['S2_DO_SUP_STR', 'S2_UP_SUP_STR', 'S1_DO_SUP_STR', 'S1_UP_SUP_STR'],
)
representative_acc = _choose_representative(
    acc_meta,
    ['S1_DO_MID_ACC_Z', 'S1_UP_MID_ACC_Z', 'S1_DO_INT_ACC_Z', 'S2_DO_MID_ACC_Z'],
)

str_signal = _actual_signal_for_display(representative_str)
acc_signal = _actual_signal_for_display(representative_acc)

FIG_AXIS_LABEL_FONTSIZE = 11
FIG_TICK_LABEL_FONTSIZE = 9

fig, axes = plt.subplots(2, 1, figsize=(8.8, 4.9), sharex=True, sharey=False, constrained_layout=False)
fig.subplots_adjust(left=0.150, right=0.985, bottom=0.150, top=0.830, hspace=0.68)
fig.suptitle('Representative STR and ACC responses under the final active window', fontsize=PLOT_TITLE_FONTSIZE, y=0.955)

panel_specs = [
    (axes[0], representative_str, str_signal, STR_COLOR, 'STR'),
    (axes[1], representative_acc, acc_signal, ACC_COLOR, 'ACC'),
]
for ax, row, display_signal, color, quantity_label in panel_specs:
    alias = row['sensor_alias']
    plot_mask = (time_grid >= x_min) & (time_grid <= x_max)
    ax.plot(time_grid, display_signal, color=color, lw=1.15, label=quantity_label)
    ax.axvspan(final_start, final_end, color='#4C78A8', alpha=0.17, linewidth=0)
    ax.axvline(final_start, color='#2563EB', lw=1.20)
    ax.axvline(final_end, color='#2563EB', lw=1.20)
    ax.set_title(f'{quantity_label}: {alias}', fontsize=PANEL_TITLE_FONTSIZE)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(*_actual_ylim(display_signal[plot_mask], display_signal[plot_mask]))
    ax.set_xlabel('Time (s)', fontsize=FIG_AXIS_LABEL_FONTSIZE)
    ax.grid(False)
    ax.tick_params(axis='both', labelsize=FIG_TICK_LABEL_FONTSIZE, length=3.0, pad=2.0)

# Units follow the AQUINAS Dataset Handbook: ACC in g (= 9.81 m/s^2), STR in per mille (= 1 mm/m).
axes[0].set_ylabel('Strain response\nchange (‰, 1 mm/m)', fontsize=FIG_AXIS_LABEL_FONTSIZE)
axes[1].set_ylabel('Acceleration response\nchange (g)', fontsize=FIG_AXIS_LABEL_FONTSIZE)
handles, labels = [], []
for ax in axes:
    h, l = ax.get_legend_handles_labels()
    handles.extend(h)
    labels.extend(l)
fig.legend(handles, labels, loc='lower center', ncol=2, frameon=True, fontsize=LEGEND_FONTSIZE)
plt.show()

print('Representative channels')
print(f"  STR = {representative_str['sensor_alias']}")
print(f"  ACC = {representative_acc['sensor_alias']}")
print(f"  final active window = {final_start:.2f} s to {final_end:.2f} s")